In [2]:
import pandas as pd
import numpy as np

In [3]:
print("numpy",np.__version__)
print("pandas",pd.__version__)

numpy 1.26.4
pandas 3.0.3


In [4]:
credits=pd.read_csv('tmdb_5000_credits.csv')
movies=pd.read_csv('tmdb_5000_movies.csv')

In [5]:
movies=movies.merge(credits,on='title')
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]
movies.dropna(inplace=True)
movies.head(3)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


In [6]:
data=movies['cast'].loc[0]
data
import ast
ast.literal_eval(data)

[{'cast_id': 242,
  'character': 'Jake Sully',
  'credit_id': '5602a8a7c3a3685532001c9a',
  'gender': 2,
  'id': 65731,
  'name': 'Sam Worthington',
  'order': 0},
 {'cast_id': 3,
  'character': 'Neytiri',
  'credit_id': '52fe48009251416c750ac9cb',
  'gender': 1,
  'id': 8691,
  'name': 'Zoe Saldana',
  'order': 1},
 {'cast_id': 25,
  'character': 'Dr. Grace Augustine',
  'credit_id': '52fe48009251416c750aca39',
  'gender': 1,
  'id': 10205,
  'name': 'Sigourney Weaver',
  'order': 2},
 {'cast_id': 4,
  'character': 'Col. Quaritch',
  'credit_id': '52fe48009251416c750ac9cf',
  'gender': 2,
  'id': 32747,
  'name': 'Stephen Lang',
  'order': 3},
 {'cast_id': 5,
  'character': 'Trudy Chacon',
  'credit_id': '52fe48009251416c750ac9d3',
  'gender': 1,
  'id': 17647,
  'name': 'Michelle Rodriguez',
  'order': 4},
 {'cast_id': 8,
  'character': 'Selfridge',
  'credit_id': '52fe48009251416c750ac9e1',
  'gender': 2,
  'id': 1771,
  'name': 'Giovanni Ribisi',
  'order': 5},
 {'cast_id': 7,
  'c

In [7]:
def convert_genres_keywords(thing):  #keyword aur genres column clean
    L=[]
    for i in ast.literal_eval(thing):
        L.append(i['name'])
    return L

def convert_cast(thing):
    # grabbing top 3 actors for the cast column
    L=[]
    cnt=0
    for item in ast.literal_eval(thing):
        if cnt>=3:
            break
        else:
            L.append(item['name'])
    return L

def convert_director(thing): #for director column
    L=[]
    for item in ast.literal_eval(thing):
        if item['job']=='Director':
            L.append(item['name'])
            break
    return L


movies['genres']=movies['genres'].apply(convert_genres_keywords)
movies['keywords']=movies['keywords'].apply(convert_genres_keywords)
movies['cast']=movies['cast'].apply(convert_cast)
movies['director']=movies['crew'].apply(convert_director)
movies['overview']=movies['overview'].apply(lambda a: a.split())

movies['genres']=movies['genres'].apply(lambda x : [item.replace(" ","") for item in x])
movies['keywords']=movies['keywords'].apply(lambda x: [item.replace(" ","") for item in x])
movies['cast']=movies['cast'].apply(lambda x: [item.replace(" ","") for item in x])
movies['director']=movies['director'].apply(lambda x: [item.replace(" ","") for item in x])

movies['tags']=movies['overview']+movies['keywords']+movies['genres']+movies['cast']+movies['director']+movies['cast']
movies['tags'] = movies['tags'].apply(lambda x: " ".join(x).lower())

movies=movies[['movie_id', 'title', 'tags']]
print(movies['tags'].loc[0])

in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d action adventure fantasy sciencefiction samworthington zoesaldana sigourneyweaver stephenlang michellerodriguez giovanniribisi joeldavidmoore cchpounder wesstudi lazalonso dileeprao mattgerald seananthonymoran jasonwhyte scottlawrence kellykilgour jamespatrickpitt seanpatrickmurphy peterdillon kevindorman kelsonhenderson davidvanhorn jacobtomuri michaelblain-rozgay joncurry lukehawker woodyschultz petermensah soniayee jahnelcurfman ilramchoi kylawarren lisaroumain debrawilson chrismala taylorkibby jodielandau julielamm cullenb.madden josephbradymadden frankietorres austinwilson sarawilson tamicawashington-miller lucybriant nathanme

In [8]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=5000,stop_words='english')
vec=cv.fit_transform(movies['tags']).toarray()
sim=cosine_similarity(vec)
sim.shape

(4806, 4806)

In [9]:
import nltk 
from nltk.stem.porter import PorterStemmer

ps=PorterStemmer()
def stem(txt):
    l=[]
    for i in txt.split(): 
        l.append(ps.stem(i))
    return " ".join(l)

movies['tags']=movies['tags'].apply(stem)
movies['tags'].loc[2]

'a cryptic messag from bond’ past send him on a trail to uncov a sinist organization. while m battl polit forc to keep the secret servic alive, bond peel back the layer of deceit to reveal the terribl truth behind spectre. spi basedonnovel secretag sequel mi6 britishsecretservic unitedkingdom action adventur crime danielcraig christophwaltz léaseydoux ralphfienn monicabellucci benwhishaw naomieharri davebautista andrewscott rorykinnear jesperchristensen alessandrocremona stephaniesigman tenochhuerta adrianapaz domenicofortunato marcozingaro stefanoelfidiclaudia ianbonar tamwilliam richardbanham pipcart simonlenagan alessandrobressanello marczinga brigittemillar adelbencherif gediminasadoma peppelanzetta francescoarca matteotaranto emilioaniba benitosagredo daitabuchi georgelasha sargonyelda andycheung erickhayden olegmirochnikov antoniosalin miloudmouradbenamara gidoschimanski nigelbarb patricenaiambana stephanecornicard garyfannin sadaoueda philliplaw waiwong josephbalderrama eijimiha

In [10]:
def recommend(mov):
    idx=movies[movies['title']==mov].index[0]
    row=sim[idx]
    lst=sorted(list(enumerate(row)),reverse=True,key=lambda x: x[1])[1:6]
    print(f"Top 5 recommendations for {mov} ':\n")
    for i in lst:
      mov_name=movies.iloc[i[0]]['title']
      score=i[1]
      per=f"{score*100:.2f}%"
      print(f"• {mov_name} — Match Score: {per}")


recommend("Cars")

Top 5 recommendations for Cars ':

• Cars 2 — Match Score: 41.28%
• A Bug's Life — Match Score: 32.00%
• Toy Story 2 — Match Score: 26.02%
• Finding Nemo — Match Score: 22.03%
• Monsters University — Match Score: 21.42%


In [ ]:
import pickle
with open('movies.pkl','wb') as f:
    pickle.dump(movies.to_dict(),f)
with open('similarity.pkl','wb') as f:
    pickle.dump(sim,f)
movie_dict = pickle.load(open('movies.pkl', 'rb'))
movies_df = pd.DataFrame(movie_dict)
movies_df
# 🫰❤️❤️


,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."
...,...,...,...
4804,9367,El Mariachi,el mariachi just want to play hi guitar and ca...
4805,72766,Newlyweds,a newlyw couple' honeymoon is upend by the arr...
4806,231617,"Signed, Sealed, Delivered","""signed, sealed, delivered"" introduc a dedic q..."
4807,126186,Shanghai Calling,when ambiti new york attorney sam is sent to s...
